# 연속형 결측 처리 대안 실험 (3번 결측 제거 없음, 로그 없음)

현재 3번은 **연속형 결측 → 중앙값(median)** 으로 채우고 있습니다. 여기서는 다른 전처리 방식을 적용했을 때 성능을 비교합니다.

**비교할 대안:**
1. **median** (기준): 현재 방식
2. **mean**: 평균으로 대체 (이상치에 민감)
3. **KNNImputer**: 비슷한 k개 샘플의 값으로 대체
4. **IterativeImputer**: 다른 변수로 결측 변수를 반복 예측 (MICE 스타일)
5. **median + 결측 표시**: 결측이었는지 0/1 컬럼 추가 후 median 대체 (모델이 "결측이었다" 정보 활용)

명목형은 기존과 동일하게 `'Missing'` 상수 대체 + OneHotEncoder 유지.

## 1. 라이브러리 및 데이터 로드

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

from pandas import read_csv

In [ ]:
PROJECT_ROOT = r'C:\itwill_bigdata_final_project-main\itwill_bigdata_final_project'
CSV_PATH = os.path.join(PROJECT_ROOT, '1. 초기 데이터 전처리', '3.coding_book_mapping.csv')

categorical_cols = [
    'w09_fam1','w09_fam2','w09edu','w09gender1','w09marital','w09edu_s','w09ecoact_s','w09enu_type',
    'w09ba069','w09bp1','w09c152','w09c001','w09c003','w09c005',
    'w09chronic_a','w09chronic_b','w09chronic_c','w09chronic_d','w09chronic_e','w09chronic_f',
    'w09chronic_g','w09chronic_h','w09chronic_i','w09chronic_j','w09chronic_k','w09chronic_l','w09chronic_m',
    'w09c056','w09c068','w09c081','w09c082','w09c085','w09c102',
    'w09smoke','w09alc','w09addic','w09c550',
    'w09f001type','w09g031',
    'w09cadd_19','w09c142','w09c143','w09c144','w09c145','w09c146','w09c147','w09c148','w09c149','w09c150','w09c151'
]

origin = read_csv(CSV_PATH, encoding='utf-8')
origin_type_changed = origin.copy()
cat_for_type = [c for c in categorical_cols if c in origin_type_changed.columns]
origin_type_changed[cat_for_type] = origin_type_changed[cat_for_type].astype('category')
origin = origin_type_changed

origin2 = origin.drop(['dependent_wage_work'], axis=1)
yname = 'dependent_ecotype'
drop_for_leakage = [yname, 'work_ability_age']
x = origin2.drop(columns=[c for c in drop_for_leakage if c in origin2.columns])
y = origin2[yname].astype(int)

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.25, random_state=52, stratify=y)
cat_cols = x_train.select_dtypes(include=['object', 'category']).columns
num_cols = x_train.select_dtypes(include=['int64', 'float64']).columns
print('train:', x_train.shape, 'test:', x_test.shape, 'num_cols:', len(num_cols), 'cat_cols:', len(cat_cols))

## 2. 전처리 대안별 파이프라인 정의 및 평가

In [ ]:
categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
    ("onehot", OneHotEncoder(drop="first", handle_unknown="ignore"))
])

def build_and_evaluate(name, numeric_pipe):
    preprocess = ColumnTransformer([
        ("num", numeric_pipe, num_cols),
        ("cat", categorical_pipe, cat_cols),
    ])
    pipe = Pipeline([
        ("preprocess", preprocess),
        ("model", LogisticRegression(random_state=52, max_iter=1000, C=0.1, class_weight='balanced'))
    ])
    pipe.fit(x_train, y_train)
    y_proba = pipe.predict_proba(x_test)[:, 1]
    auc = roc_auc_score(y_test, y_proba)
    cv_auc = cross_val_score(pipe, x_train, y_train, cv=5, scoring='roc_auc', n_jobs=-1).mean()
    return {'전처리': name, 'Test_AUC': round(auc, 4), 'CV5_AUC': round(cv_auc, 4)}

results_list = []

# 1) median (기준)
numeric_median = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
results_list.append(build_and_evaluate('median(기준)', numeric_median))

# 2) mean
numeric_mean = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler())
])
results_list.append(build_and_evaluate('mean', numeric_mean))

# 3) KNNImputer (k=5)
numeric_knn = Pipeline([
    ("imputer", KNNImputer(n_neighbors=5, weights='distance')),
    ("scaler", StandardScaler())
])
results_list.append(build_and_evaluate('KNNImputer(k=5)', numeric_knn))

# 4) IterativeImputer (MICE 스타일)
numeric_iter = Pipeline([
    ("imputer", IterativeImputer(max_iter=10, random_state=52)),
    ("scaler", StandardScaler())
])
results_list.append(build_and_evaluate('IterativeImputer', numeric_iter))

# 5) median + 결측 표시: 결측이었던 위치를 0/1 컬럼으로 추가 후 median 대체
from sklearn.base import BaseEstimator, TransformerMixin

class MedianImputerWithMissingIndicator(BaseEstimator, TransformerMixin):
    """연속형 결측을 중앙값으로 채우고, 결측 여부 0/1 컬럼을 붙인 뒤 스케일."""
    def __init__(self):
        self.imputer = SimpleImputer(strategy="median")
        self.scaler = StandardScaler()

    def fit(self, X, y=None):
        self.imputer.fit(X)
        X_imputed = self.imputer.transform(X)
        self.scaler.fit(X_imputed)
        return self

    def transform(self, X):
        X_imputed = self.imputer.transform(X)
        missing = np.isnan(X).astype(np.float64)
        scaled = self.scaler.transform(X_imputed)
        return np.hstack([scaled, missing])

    def get_feature_names_out(self, input_features=None):
        if input_features is None:
            n = self.imputer.statistics_.shape[0]
            input_features = [f"x{i}" for i in range(n)]
        out = list(input_features) + [f"{c}_is_missing" for c in input_features]
        return np.array(out)

numeric_median_indicator = Pipeline([
    ("imputer_with_flag", MedianImputerWithMissingIndicator()),
])
results_list.append(build_and_evaluate('median+결측표시', numeric_median_indicator))

df_results = pd.DataFrame(results_list)
display(df_results)

## 3. 요약 및 추가 대안 안내

In [ ]:
print('전처리 대안별 Test AUC (로지스틱, 동일 하이퍼파라미터)')
print(df_results.to_string(index=False))
best = df_results.loc[df_results['Test_AUC'].idxmax()]
print(f"\n가장 높은 Test AUC: {best['전처리']} ({best['Test_AUC']})")

### 참고: 그 외 시도해 볼 수 있는 방안

- **컬럼별 전략 분리**: 소득/자산은 0 또는 median, 나이는 mean 등 변수 성격에 따라 다르게 지정
- **KNNImputer k 변경**: k=3, 10 등으로 실험
- **IterativeImputer max_iter, estimator 변경**: 기본 BayesianRidge 대신 다른 회귀 사용
- **트리 모델만 결측 유지**: CatBoost/LightGBM/XGBoost는 결측을 그대로 두고 학습 가능 (해당 노트북에서만 연속형 Imputer 제거 후 실험)